In [1]:
import os
import pandas as pd
from glob import glob
import numpy as np
import geopandas as gpd

os.chdir('/store/carroll/sbgplants/')

In [13]:
# file paths
raw = 'data/raw'

out_folder = 'data/out_csv'

table = 'raster_plot_event'

In [3]:
# load relevant data

fid_boundary = gpd.read_file('/store/carroll/col/data/2018/raw/crbu_2018_fid_boundaries.geojson')
insitu_plot_event = pd.read_csv(os.path.join(out_folder, 'insitu_plot_event.csv'))
insitu_plot_shape = gpd.read_file(os.path.join(out_folder, 'insitu_plot_shape.geojson'))

In [11]:
out_table = insitu_plot_shape.copy()

out_table = out_table.merge(insitu_plot_event, on='insitu_plot_event_id', how='left')

out_table = out_table[['plot_name', 'collection_date_x', 'geometry']]
out_table = out_table.rename(columns={'collection_date_x':'sample_date'})
out_table['extraction_method'] = None

out_table = gpd.sjoin(
    out_table,
    fid_boundary[['fid', 'geometry']],
    how='left',
    predicate='within' # where the plot is within the fid boundary
).drop(columns=['index_right'])

out_table['raster_plot_id'] = range(len(out_table))
out_table = out_table.rename(columns={'fid':'granule_id'})
out_table = out_table[['raster_plot_id', 'plot_name', 'sample_date', 'geometry', 'granule_id', 'extraction_method']]

out_table

,raster_plot_id,plot_name,sample_date,geometry,granule_id,extraction_method
0,0,325-ER18,2018-06-25,"MULTIPOLYGON (((327909.079 4313992.026, 327912...",NIS01_20180613_172129,None
0,1,325-ER18,2018-06-25,"MULTIPOLYGON (((327909.079 4313992.026, 327912...",NIS01_20180613_173216,None
0,2,325-ER18,2018-06-25,"MULTIPOLYGON (((327909.079 4313992.026, 327912...",NIS01_20180613_174241,None
0,3,325-ER18,2018-06-25,"MULTIPOLYGON (((327909.079 4313992.026, 327912...",NIS01_20180619_160339,None
0,4,325-ER18,2018-06-25,"MULTIPOLYGON (((327909.079 4313992.026, 327912...",NIS01_20180625_165243,None
...,...,...,...,...,...,...
460,2148,478-ER18,2018-07-30,"MULTIPOLYGON (((322940.93 4303914.041, 322943....",NIS01_20180621_173138,None
460,2149,478-ER18,2018-07-30,"MULTIPOLYGON (((322940.93 4303914.041, 322943....",NIS01_20180625_171533,None
460,2150,478-ER18,2018-07-30,"MULTIPOLYGON (((322940.93 4303914.041, 322943....",NIS01_20180625_172518,None
460,2151,478-ER18,2018-07-30,"MULTIPOLYGON (((322940.93 4303914.041, 322943....",NIS01_20180625_173511,None


In [12]:
# export table
fp_out = os.path.join(out_folder, f'{table}.geojson')
out_table.to_file(fp_out, index=False)